In [ ]:
### Programs
import os, sys
os.environ['USE_PYGEOS'] = '0'

import regionmask
import geopandas as gpd
import shapely
import xarray as xr
from shapely.geometry import Polygon
from shapely.geometry import Point
from shapely.ops import cascaded_union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Directories
topdir             = os.path.join(os.path.expanduser("~"), "Dropbox/Projects/")
projectdir         = os.path.join(topdir, "Maize_prediction")
cropdir            = os.path.join(topdir, "Crop_misallocation")
datadir            = os.path.join(projectdir,"Data")
amcadir            = os.path.join(datadir,"INEGI","Areas_Censal_Agropecuario_2016")

### Output
census_areas       = os.path.join(amcadir,"census_areas.shp")
agebs_fromadc      = os.path.join(amcadir,"agebs_from_ADC16.shp")
mun_fromadc_areas  = os.path.join(amcadir,"muns_from_ADC16_areas.shp")

In [2]:
totalgdf = gpd.GeoDataFrame()
for i in list(range(1,33))[:]:
    if i < 10: 
        st_code = "0"+str(i)
    else:
        st_code = str(i)
        
    st_path = os.path.join(amcadir,"ac_"+st_code,"Proyecto MNA_"+st_code,"Cartografia","ac_"+st_code+".shp")
    gdf = gpd.read_file(st_path)
    gdf = gdf.to_crs('epsg:4326')
    totalgdf = totalgdf.append(gdf, sort=False)

### Write to ADC shapefile
totalgdf.to_file(census_areas,index=False)

/tmp/ipykernel_51331/35209180.py:11: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  totalgdf = totalgdf.append(gdf, sort=False)


In [4]:
### Write to AGEB shapefile
totalgdf                        =  gpd.read_file(census_areas,index=False)
totalgdf.loc[:,'ageb']          =  totalgdf['CONTROL'].apply(lambda x: x[:10])
totalgdf.loc[128925,'geometry'] =  totalgdf.loc[128925,'geometry'].simplify(tolerance=0.0121225)
ageb_df_from_adc                =  totalgdf.dissolve(by='ageb').reset_index()
ageb_df_from_adc[['ageb','geometry']].to_file(agebs_fromadc,index=False)


In [13]:
### Write to mun shapefile
totalgdf                        =  gpd.read_file(census_areas,index=False)
totalgdf.loc[:,'muncode']       =  totalgdf['CONTROL'].apply(lambda x: x[:5])
totalgdf.loc[128925,'geometry'] =  totalgdf.loc[128925,'geometry'].simplify(tolerance=0.0121225)
mundf_from_adc                  =  totalgdf.dissolve(by='muncode').reset_index()
mundf_from_adc[['muncode','geometry']].to_file(mun_fromadc_areas, index=False)
